In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

---

In [5]:
# Step 1: Create Dataset

data = pd.DataFrame({
    
    "VehicleCount": [
        100, 120, 150, 180, 200,
        400, 450, 500, 550, 600,
        250, 300
    ],
    "AverageSpeed": [
        55, 52, 50, 48, 45,
        35, 30, 28, 25, 20,
        42, 40
    ],
    "NumberOfSignals": [
        2, 2, 3, 3, 4,
        5, 5, 6, 6, 7,
        4, 4
    ],
    "TimeOfDay": [
        8, 9, 10, 11, 12,
        17, 18, 19, 20, 21,
        13, 14
    ],
    "WeatherScore": [
        90, 85, 88, 80, 82,
        75, 70, 65, 60, 55,
        78, 80
    ],
    "Congestion": [
        0, 0, 0, 0, 0,
        1, 1, 1, 1, 1,
        0, 0
    ]
})

In [6]:
print("\nOriginal Dataset:\n")
data


Original Dataset:



,VehicleCount,AverageSpeed,NumberOfSignals,TimeOfDay,WeatherScore,Congestion
0,100,55,2,8,90,0
1,120,52,2,9,85,0
2,150,50,3,10,88,0
3,180,48,3,11,80,0
4,200,45,4,12,82,0
5,400,35,5,17,75,1
6,450,30,5,18,70,1
7,500,28,6,19,65,1
8,550,25,6,20,60,1
9,600,20,7,21,55,1


---

In [7]:
# Step 2: Separate Features and Target

X = data.drop("Congestion", axis=1)

y = data["Congestion"]

---

In [8]:
# Step 3: Handle Missing Values

imputer = SimpleImputer(strategy="mean")

X_imputed = imputer.fit_transform(X)

---

In [9]:
# Step 4: Standardize Features

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

---

In [10]:
# Step 5: K-Means Clustering

kmeans = KMeans(n_clusters=3, random_state=42)

clusters = kmeans.fit_predict(X_scaled)

In [11]:
# Add cluster to original dataset

data["TrafficCluster"] = clusters

print("\nTraffic Clusters:\n")

print(data[["VehicleCount", "AverageSpeed", "TrafficCluster"]])


Traffic Clusters:

    VehicleCount  AverageSpeed  TrafficCluster
0            100            55               2
1            120            52               2
2            150            50               2
3            180            48               2
4            200            45               0
5            400            35               0
6            450            30               1
7            500            28               1
8            550            25               1
9            600            20               1
10           250            42               0
11           300            40               0


---

In [12]:
# Step 6: Add Cluster as New Feature

X_final = pd.DataFrame(X_scaled, columns=["VehicleCount", "AverageSpeed", "NumberOfSignals", "TimeOfDay", "WeatherScore"])

X_final["TrafficCluster"] = clusters

---

In [13]:
# Step 7: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.25, random_state=42, stratify=y)

---

In [14]:
# Step 8: Train Random Forest

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


---

In [15]:
# Step 9: Make Predictions

y_pred = model.predict(X_test)

---

In [16]:
# Step 10: Evaluate Model

accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy}")


Accuracy: 1.0


---

In [19]:
# Step 11: Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")

print(cm)


Confusion Matrix:
[[2 0]
 [0 1]]


---